In [1]:
"""Flat PEPS boundary-contraction notebook experiments."""
from time import perf_counter
from IPython.display import display
import quimb.tensor as qtn

import pepsy as py
core = py.core

import torch


In [ ]:
to_backend = core.backend_torch(dtype=torch.complex128)
optimizer = core.build_optimizer(progbar=False, directory="cash/", parallel=True)


In [3]:
grid_x, grid_y = 10, 10
partition_state = qtn.TN2D_classical_ising_partition_function(Lx=grid_x, Ly=grid_y, beta=0.4)
partition_state.expand_bond_dimension_(5, rand_strength=2)


TensorNetwork2D(tensors=100, indices=180, Lx=10, Ly=10, max_bond=5)

In [4]:
partition_state.show()


    5    5    5    5    5    5    5    5    5   
 ●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●
 ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5  
 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃ 
 ●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●
 ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5  
 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃ 
 ●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●
 ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5  
 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃ 
 ●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●
 ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5  
 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃ 
 ●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●
 ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5  
 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃ 
 ●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●
 ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5   ┃5  
 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃  5 ┃ 
 ●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●━━━━●
 ┃5   ┃5   ┃5  

In [5]:
# build_bra_ket API smoke test (norm + mps_boundaries)
partition_state.apply_to_arrays(to_backend)


In [6]:
display(partition_state)


TensorNetwork2D(tensors=100, indices=180, Lx=10, Ly=10, max_bond=5)

In [8]:
# Return boundary MPSs.
# pylint: disable=unexpected-keyword-arg
boundary_mps = py.BdyMPS(
    tn_flat=partition_state,
    tn_double=partition_state,
    chi=20,
    single_layer=False,
    flat=True,
)


In [9]:
# Retune all boundary MPS bonds to target chi (expand or compress as needed).
boundary_mps.expand_bnd(32)
boundary_mps.normalize()
display((boundary_mps.norm, boundary_mps.chi))


(tensor(1.+4.1093e-18j, dtype=torch.complex128), 32)

In [10]:
boundary_mps.show("Y2_l")


grid cut=Y2 (direction=y, side=left)
Y9  ○──○──○──○──○──○──○──○──○──○
    │  │  │  │  │  │  │  │  │  │
Y8  ○──○──○──○──○──○──○──○──○──○
    │  │  │  │  │  │  │  │  │  │
Y7  ○──○──○──○──○──○──○──○──○──○
    │  │  │  │  │  │  │  │  │  │
Y6  ○──○──○──○──○──○──○──○──○──○
    │  │  │  │  │  │  │  │  │  │
Y5  ○──○──○──○──○──○──○──○──○──○
    │  │  │  │  │  │  │  │  │  │
Y4  ○──○──○──○──○──○──○──○──○──○
    │  │  │  │  │  │  │  │  │  │
Y3  ○──○──○──○──○──○──○──○──○──○
    │  │  │  │  │  │  │  │  │  │
Y2  ●> ●> ●> ●> ●> ●> ●> ●> ●> ●
    │  │  │  │  │  │  │  │  │  │
Y1  ○──○──○──○──○──○──○──○──○──○
    │  │  │  │  │  │  │  │  │  │
Y0  ○──○──○──○──○──○──○──○──○──○
    X0 X1 X2 X3 X4 X5 X6 X7 X8 X9

│5│25│32│32│32│32│32│32│32│
>─>──>──>──>──>──>──>──>──●
│ │  │  │  │  │  │  │  │  │


In [11]:
print(boundary_mps.mps_b["Y0_l"].norm())
boundary_mps.mps_b["Y0_l"].show()


tensor(1.0000+0.j, dtype=torch.complex128)
 5 25 32 32 32 32 32 32 32 
>─>──>──>──>──>──>──>──>──●
│ │  │  │  │  │  │  │  │  │


In [12]:
# Retune all boundary MPS bonds to target chi (expand or compress as needed).
boundary_mps.expand_bnd(120)
boundary_mps.normalize()
display((boundary_mps.norm, boundary_mps.chi))


(tensor(1.-4.1375e-17j, dtype=torch.complex128), 120)

In [13]:
boundary_mps.show("Y1_r")


grid cut=Y8 (direction=y, side=right)
Y9  ○──○──○──○──○──○──○──○──○──○
    │  │  │  │  │  │  │  │  │  │
Y8  ● <● <● <● <● <● <● <● <● <●
    │  │  │  │  │  │  │  │  │  │
Y7  ○──○──○──○──○──○──○──○──○──○
    │  │  │  │  │  │  │  │  │  │
Y6  ○──○──○──○──○──○──○──○──○──○
    │  │  │  │  │  │  │  │  │  │
Y5  ○──○──○──○──○──○──○──○──○──○
    │  │  │  │  │  │  │  │  │  │
Y4  ○──○──○──○──○──○──○──○──○──○
    │  │  │  │  │  │  │  │  │  │
Y3  ○──○──○──○──○──○──○──○──○──○
    │  │  │  │  │  │  │  │  │  │
Y2  ○──○──○──○──○──○──○──○──○──○
    │  │  │  │  │  │  │  │  │  │
Y1  ○──○──○──○──○──○──○──○──○──○
    │  │  │  │  │  │  │  │  │  │
Y0  ○──○──○──○──○──○──○──○──○──○
    X0 X1 X2 X3 X4 X5 X6 X7 X8 X9

│5│25│120│120│120│120│120│120│120│
>─>──>━━━>━━━>━━━>━━━>━━━>━━━>━━━●
│ │  │   │   │   │   │   │   │   │


In [14]:
contract_value = partition_state.contract(all, optimize=optimizer)
print(complex(contract_value))


F=10.51 C=10.90 S=25.54 P=26.54: 100%|██████████| 64/64 [00:19<00:00,  3.25it/s]      


(-5.352479742709316e+91+0j)


In [15]:
boundary_result = py.contract_boundary(
    norm=partition_state,
    mps_boundaries=boundary_mps.mps_b,
    contraction_opt=optimizer,
    n_iter=1,
    progress=True,
    direction="y",
    max_separation=0,
    track_boundary_fidelity=True,
    flat=True,
)

display(boundary_result)


bdy_dmrg:: 100%|██████████| 8/8 [00:17<00:00,  2.14s/it, chi=120, F=0.052]


BoundaryContractResult(cost=tensor(-1.7563e+91+1.2665e+91j, dtype=torch.complex128), fidel=[1.0000000000000007, 0.9919735302074479, 0.5489341012816141, 0.4116772858152396, 1.000000000000001, 0.9923842637236989, 0.5758018979391201, 0.4058685917991901], direction='y', n_iter=1, max_separation=0)

In [16]:
display(boundary_mps.norm)


tensor(2.0276e+46-3.6718e+28j, dtype=torch.complex128)

In [17]:
start_time = perf_counter()
reference_value = partition_state.contract_boundary(
    max_bond=20,
    mode="mps",
    final_contract_opts={"optimize": optimizer},
    cutoff=1e-14,
    progbar=True,
    layer_tags=["KET", "BRA"],
    max_separation=1,
)
print(f"contract_boundary elapsed: {perf_counter() - start_time:.3f}s")
display(reference_value)


contracted boundary, Lx=2, Ly=10: : 8it [00:00, 19.36it/s]
F=5.70 C=6.18 S=10.97 P=14.74:   0%|          | 0/64 [00:00<?, ?it/s]

contract_boundary elapsed: 0.549s


tensor(-4.6898e+90+0.j, dtype=torch.complex128)

In [18]:
cost_result = py.contract_boundary(
    norm=partition_state,
    mps_boundaries=boundary_mps.mps_b,
    contraction_opt=optimizer,
    n_iter=5,
    progress=True,
    track_boundary_fidelity=True,
    visualize=False,
    direction="y",
    retag=True,
    max_separation=0,
    flat=True,
)

display(cost_result)


bdy_dmrg:: 100%|██████████| 8/8 [00:08<00:00,  1.12s/it, chi=120, F=0.285]


BoundaryContractResult(cost=tensor(-5.9225e+91-2.0462e+91j, dtype=torch.complex128), fidel=[0.9999999999999996, 0.9990086283223426, 0.8002990986771664, 0.6632845481443503, 1.0000000000000002, 0.9991307585256979, 0.8179699887612126, 0.6583543868832504], direction='y', n_iter=5, max_separation=0)

In [19]:
display(boundary_mps.norm)


tensor(3.1021e+46-8.9496e+28j, dtype=torch.complex128)